# NB_lab_test_trends
**Target:** `caresyncdatabricks.gold.lab_test_trends`  |  **Grain:** one row per test type (`test_name`) per calendar month  |  **Load:** FULL (CREATE OR REPLACE TABLE ... AS SELECT)

**Driving table:** `silver.lab_results` grouped by `test_name`, `year(test_date)`, `month(test_date)` (no filters)
* LEFT JOIN calendar on the first day of the month (`DateKey = yyyyMM01`) for `month_name` / `quarter_name`
* Window: `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month`

Note: the physical calendar table is `silver.calender` (PascalCase columns). It has no `quarter_name` column, so `quarter_name` is derived as `'Q' || Quarter` (e.g. "Q1").

In [0]:
%sql
CREATE OR REPLACE TABLE caresyncdatabricks.gold.lab_test_trends AS
WITH monthly AS (
  SELECT
    test_name,
    YEAR(test_date)                      AS year,
    MONTH(test_date)                     AS month,
    COUNT(*)                             AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM caresyncdatabricks.silver.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  r.test_name,
  CAST(r.year AS INT)                                                       AS year,
  CAST(r.month AS INT)                                                      AS month,
  c.MonthName                                                               AS month_name,
  CONCAT('Q', c.Quarter)                                                    AS quarter_name,
  CAST(r.total_tests AS INT)                                                AS total_tests,
  CAST(r.abnormal_test_count AS INT)                                        AS abnormal_test_count,
  CAST(r.critical_test_count AS INT)                                        AS critical_test_count,
  r.abnormal_rate_pct,
  r.critical_rate_pct,
  CAST(r.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                      AS prev_month_abnormal_rate_pct,
  CAST(r.abnormal_rate_pct - r.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag r
LEFT JOIN caresyncdatabricks.silver.calender c
  ON c.DateKey = CAST(date_format(make_date(r.year, r.month, 1), 'yyyyMMdd') AS INT)

In [0]:
updated_df = spark.table("caresyncdatabricks.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))